In [2]:
import numpy as np
import json
from transformers import AutoTokenizer, AutoConfig
from transformers import BertPreTrainedModel, BertModel
from tqdm import tqdm, tqdm_notebook
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from scipy.special import softmax

In [3]:
# The performance of adolescent bss srf-llm (triplet classification model to assess 3-level suicidal risk) on training dataset.
base = 'D:\\MyJupy\\SRF-LLM_Verification\\'
# Set the base directory.

hid_dim = 11
bat_size = 5
data_dir = base + 'adolescent_bss\\'
linear_model = np.load(data_dir + 'Adolescent_BSS_Suicide_Linear_model_SegTime_18_weights.npy')
# Load the trained linear layer.

devi = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using {devi} device')
trained_model = data_dir + 'Adolescent_BSS_Suicide_Bert_Pre3min_SegTime_16_Epoch_69_weights.bin'
lab = np.loadtxt(data_dir + 'Adolescent_train_lable.txt',dtype=str).astype(np.float32)

class dFC(Dataset):
    def __init__(self, data_file):
        self.data = self.load_data(data_file)
    
    def load_data(self, data_file):
        Data = {}
        with open(data_file, 'rt', encoding='utf-8') as f:
            samples = json.load(f)
            for idx, sample in enumerate(samples):
                Data[idx] = samples[sample]
        return Data
    
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

data_1 = dFC(data_dir + 'Adolescent_train_dat.json')
print(f"------There are {len(data_1)} samples------\n")

class BertForSuicide(BertPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.bert = BertModel(config, add_pooling_layer=False)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(768, hid_dim)
        self.post_init()
    
    def forward(self, x):
        bert_output = self.bert(**x)
        cls_vectors = bert_output.last_hidden_state[:, 0, :]
        cls_vectors = self.dropout(cls_vectors)
        logits = self.classifier(cls_vectors)
        return logits

model_path = base + 'bert_base_chinese\\'
tokenizer = AutoTokenizer.from_pretrained(model_path)
default_confi = AutoConfig.from_pretrained(model_path)
default_model = BertForSuicide.from_pretrained(model_path, config=default_confi).to(devi)
state = torch.load(trained_model)
default_model.load_state_dict(state)
# Load the trained Bert layer.

def collote_fn(batch_samples):
    batch_sentence = []
    for sample in batch_samples:
        batch_sentence.append(sample['sentence'])
    X = tokenizer(
        batch_sentence, 
        padding=True, 
        truncation=True, 
        return_tensors="pt"
    ).to(devi)
    return X

def linear_acc_auc(weigh,hidd,lab_norm):
    intercept = weigh[:,0].T
    coef = weigh[:,1:].T
    lab = hidd@coef+intercept
    lab_soft = softmax(lab,axis=1)
    lab_cat = np.argmax(lab_soft,axis=1)
    acc = np.array([1. for (p,n) in zip(lab_cat,lab_norm) if p == n]).sum()
    acc /= lab_norm.shape[0]
    auc = roc_auc_score(lab_norm,lab_soft,average='weighted',multi_class='ovo')
    return acc,auc

def session_test(ses,labb):
    test_dataloader = DataLoader(ses, batch_size=bat_size, shuffle=False, collate_fn=collote_fn)
    hid_predicted = np.zeros([2,hid_dim],dtype=np.float32)
    default_model.eval()
    with torch.no_grad():
        print('evaluating on test set...')
        for batch_data in tqdm(test_dataloader):
            batch_data = batch_data.to(devi)
            pred = default_model(batch_data)
            hid_predicted = np.concatenate((hid_predicted,np.array([element.item() for element in pred.reshape([1,-1]).squeeze()]).reshape([-1,pred.shape[-1]])))

    acc_test, auc_test = linear_acc_auc(linear_model,hid_predicted[2:,:],labb)
    print(f"------Test Acc: {(100*acc_test):>0.5f}%------\n")
    print(f"------Test Auc: {auc_test:>0.7f}------\n")
    
session_test(data_1,lab)

Some weights of BertForSuicide were not initialized from the model checkpoint at D:\MyJupy\work_1\pilou\bert_base_chinese\ and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using cuda device
------There are 100 samples------

evaluating on test set...


100%|██████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 22.59it/s]

------Test Acc: 94.00000%------

------Test Auc: 0.9676736------



In [4]:
# The performance of adolescent bss srf-llm (triplet classification model to assess 3-level suicidal risk) on test dataset.
base = 'D:\\MyJupy\\SRF-LLM_Verification\\'
# Set the base directory.

hid_dim = 11
bat_size = 5
data_dir = base + 'adolescent_bss\\'
linear_model = np.load(data_dir + 'Adolescent_BSS_Suicide_Linear_model_SegTime_18_weights.npy')
# Load the trained linear layer.

devi = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using {devi} device')
trained_model = data_dir + 'Adolescent_BSS_Suicide_Bert_Pre3min_SegTime_16_Epoch_69_weights.bin'
lab = np.loadtxt(data_dir + 'Adolescent_test_lable.txt',dtype=str).astype(np.float32)

class dFC(Dataset):
    def __init__(self, data_file):
        self.data = self.load_data(data_file)
    
    def load_data(self, data_file):
        Data = {}
        with open(data_file, 'rt', encoding='utf-8') as f:
            samples = json.load(f)
            for idx, sample in enumerate(samples):
                Data[idx] = samples[sample]
        return Data
    
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

data_1 = dFC(data_dir + 'Adolescent_test_dat.json')
print(f"------There are {len(data_1)} samples------\n")

class BertForSuicide(BertPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.bert = BertModel(config, add_pooling_layer=False)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(768, hid_dim)
        self.post_init()
    
    def forward(self, x):
        bert_output = self.bert(**x)
        cls_vectors = bert_output.last_hidden_state[:, 0, :]
        cls_vectors = self.dropout(cls_vectors)
        logits = self.classifier(cls_vectors)
        return logits

model_path = base + 'bert_base_chinese\\'
tokenizer = AutoTokenizer.from_pretrained(model_path)
default_confi = AutoConfig.from_pretrained(model_path)
default_model = BertForSuicide.from_pretrained(model_path, config=default_confi).to(devi)
state = torch.load(trained_model)
default_model.load_state_dict(state)
# Load the trained Bert layer.

def collote_fn(batch_samples):
    batch_sentence = []
    for sample in batch_samples:
        batch_sentence.append(sample['sentence'])
    X = tokenizer(
        batch_sentence, 
        padding=True, 
        truncation=True, 
        return_tensors="pt"
    ).to(devi)
    return X

def linear_acc_auc(weigh,hidd,lab_norm):
    intercept = weigh[:,0].T
    coef = weigh[:,1:].T
    lab = hidd@coef+intercept
    lab_soft = softmax(lab,axis=1)
    lab_cat = np.argmax(lab_soft,axis=1)
    acc = np.array([1. for (p,n) in zip(lab_cat,lab_norm) if p == n]).sum()
    acc /= lab_norm.shape[0]
    auc = roc_auc_score(lab_norm,lab_soft,average='weighted',multi_class='ovo')
    return acc,auc

def session_test(ses,labb):
    test_dataloader = DataLoader(ses, batch_size=bat_size, shuffle=False, collate_fn=collote_fn)
    hid_predicted = np.zeros([2,hid_dim],dtype=np.float32)
    default_model.eval()
    with torch.no_grad():
        print('evaluating on test set...')
        for batch_data in tqdm(test_dataloader):
            batch_data = batch_data.to(devi)
            pred = default_model(batch_data)
            hid_predicted = np.concatenate((hid_predicted,np.array([element.item() for element in pred.reshape([1,-1]).squeeze()]).reshape([-1,pred.shape[-1]])))

    acc_test, auc_test = linear_acc_auc(linear_model,hid_predicted[2:,:],labb)
    print(f"------Test Acc: {(100*acc_test):>0.5f}%------\n")
    print(f"------Test Auc: {auc_test:>0.7f}------\n")
    
session_test(data_1,lab)

Some weights of BertForSuicide were not initialized from the model checkpoint at D:\MyJupy\work_1\pilou\bert_base_chinese\ and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using cuda device
------There are 37 samples------

evaluating on test set...


100%|████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:00<00:00, 19.79it/s]

------Test Acc: 56.75676%------

------Test Auc: 0.6763247------



In [5]:
# The performance of adolescent bss srf-llm (triplet classification model to assess 3-level suicidal risk) on generalization dataset (with three sessions of rsfMRI data collection).
base = 'D:\\MyJupy\\SRF-LLM_Verification\\'
# Set the base directory.

hid_dim = 11
bat_size = 5
data_dir = base + 'adolescent_bss\\'
linear_model = np.load(data_dir + 'Adolescent_BSS_Suicide_Linear_model_SegTime_18_weights.npy')
# Load the trained linear layer.

devi = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using {devi} device')
trained_model = data_dir + 'Adolescent_BSS_Suicide_Bert_Pre3min_SegTime_16_Epoch_69_weights.bin'
lab = np.loadtxt(data_dir + 'Adolescent_generalization_lable.txt',dtype=str).astype(np.float32)

class dFC(Dataset):
    def __init__(self, data_file):
        self.data = self.load_data(data_file)
    
    def load_data(self, data_file):
        Data = {}
        with open(data_file, 'rt', encoding='utf-8') as f:
            samples = json.load(f)
            for idx, sample in enumerate(samples):
                Data[idx] = samples[sample]
        return Data
    
    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

data_1 = dFC(data_dir + 'Adolescent_generalization_dat_1.json')
data_2 = dFC(data_dir + 'Adolescent_generalization_dat_2.json')
data_3 = dFC(data_dir + 'Adolescent_generalization_dat_3.json')
print(f"------There are {len(data_1)} samples------\n")

class BertForSuicide(BertPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.bert = BertModel(config, add_pooling_layer=False)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(768, hid_dim)
        self.post_init()
    
    def forward(self, x):
        bert_output = self.bert(**x)
        cls_vectors = bert_output.last_hidden_state[:, 0, :]
        cls_vectors = self.dropout(cls_vectors)
        logits = self.classifier(cls_vectors)
        return logits

model_path = base + 'bert_base_chinese\\'
tokenizer = AutoTokenizer.from_pretrained(model_path)
default_confi = AutoConfig.from_pretrained(model_path)
default_model = BertForSuicide.from_pretrained(model_path, config=default_confi).to(devi)
state = torch.load(trained_model)
default_model.load_state_dict(state)
# Load the trained Bert layer.

def collote_fn(batch_samples):
    batch_sentence = []
    for sample in batch_samples:
        batch_sentence.append(sample['sentence'])
    X = tokenizer(
        batch_sentence, 
        padding=True, 
        truncation=True, 
        return_tensors="pt"
    ).to(devi)
    return X

def linear_acc_auc(weigh,hidd,lab_norm):
    intercept = weigh[:,0].T
    coef = weigh[:,1:].T
    lab = hidd@coef+intercept
    lab_soft = softmax(lab,axis=1)
    lab_cat = np.argmax(lab_soft,axis=1)
    acc = np.array([1. for (p,n) in zip(lab_cat,lab_norm) if p == n]).sum()
    acc /= lab_norm.shape[0]
    auc = roc_auc_score(lab_norm,lab_soft,average='weighted',multi_class='ovo')
    return acc,auc

def session_test(ses,labb):
    test_dataloader = DataLoader(ses, batch_size=bat_size, shuffle=False, collate_fn=collote_fn)
    hid_predicted = np.zeros([2,hid_dim],dtype=np.float32)
    default_model.eval()
    with torch.no_grad():
        print('evaluating on test set...')
        for batch_data in tqdm(test_dataloader):
            batch_data = batch_data.to(devi)
            pred = default_model(batch_data)
            hid_predicted = np.concatenate((hid_predicted,np.array([element.item() for element in pred.reshape([1,-1]).squeeze()]).reshape([-1,pred.shape[-1]])))

    acc_test, auc_test = linear_acc_auc(linear_model,hid_predicted[2:,:],labb)
    print(f"------Test Acc: {(100*acc_test):>0.5f}%------\n")
    print(f"------Test Auc: {auc_test:>0.7f}------\n")
    
session_test(data_1,lab)
session_test(data_2,lab)
session_test(data_3,lab)

Using cuda device
------There are 271 samples------



Some weights of BertForSuicide were not initialized from the model checkpoint at D:\MyJupy\work_1\pilou\bert_base_chinese\ and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


evaluating on test set...


100%|██████████████████████████████████████████████████████████████████████████████████| 55/55 [00:01<00:00, 38.85it/s]


------Test Acc: 42.80443%------

------Test Auc: 0.5347020------

evaluating on test set...


100%|██████████████████████████████████████████████████████████████████████████████████| 55/55 [00:01<00:00, 44.07it/s]


------Test Acc: 46.12546%------

------Test Auc: 0.5856226------

evaluating on test set...


100%|██████████████████████████████████████████████████████████████████████████████████| 55/55 [00:01<00:00, 42.86it/s]

------Test Acc: 43.54244%------

------Test Auc: 0.5261573------

